<a href="https://colab.research.google.com/github/muntakson/omics-atlas-workshop/blob/main/notebooks/04_interpreting_enrichment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 Workshop 4 — Interpreting & visualizing enrichment

**Companion to [Chapter 4: Interpreting Omics Data with Pathway Enrichment](https://omics.iotok.org/chapters/interpreting-omics-enrichment)**.

Enriched pathways are often redundant. The **EnrichmentMap** idea (Chapter 3/4) draws pathways as a *network*: nodes = pathways, edges connect pathways that **share genes**. Clusters reveal the real themes.

In [ ]:
!pip install -q gseapy

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, networkx as nx
import gseapy as gp
RAW = 'https://raw.githubusercontent.com/muntakson/omics-atlas-workshop/main'
res = pd.read_csv(f'{RAW}/data/airway_deseq2_results.csv').dropna(subset=['symbol','padj'])
sig = res[(res.padj < 0.05) & (res.log2FoldChange.abs() > 1)]
genes = sig['symbol'].tolist()
print(len(genes), 'DEGs')

## 1. Get enriched terms with their gene sets

In [ ]:
enr = gp.enrichr(gene_list=genes, gene_sets=['GO_Biological_Process_2021'],
                 organism='human', outdir=None).results
enr = enr[enr['Adjusted P-value'] < 0.1].copy()
enr['geneset'] = enr['Genes'].str.split(';').apply(set)
print(len(enr), 'significant terms')

## 2. Build the enrichment map
Edge weight = **Jaccard overlap** of the two pathways' genes. We keep edges above a threshold, then draw.

In [ ]:
terms = enr.sort_values('Adjusted P-value').head(25).reset_index(drop=True)
G = nx.Graph()
for i, row in terms.iterrows():
    G.add_node(i, label=row['Term'].split(' (GO')[0][:32], size=len(row['geneset']))
for i in range(len(terms)):
    for j in range(i+1, len(terms)):
        a, b = terms.loc[i,'geneset'], terms.loc[j,'geneset']
        jac = len(a & b) / len(a | b)
        if jac > 0.25:
            G.add_edge(i, j, weight=jac)
print(G.number_of_nodes(), 'nodes', G.number_of_edges(), 'edges')

In [ ]:
plt.figure(figsize=(9,7))
pos = nx.spring_layout(G, seed=7, k=0.6)
sizes = [G.nodes[n]['size']*40 for n in G]
nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color='#12a594', alpha=.8)
nx.draw_networkx_edges(G, pos, alpha=.3, width=[G[u][v]['weight']*4 for u,v in G.edges])
nx.draw_networkx_labels(G, pos, labels={n:G.nodes[n]['label'] for n in G}, font_size=7)
plt.title('Enrichment map — airway dexamethasone response'); plt.axis('off'); plt.tight_layout(); plt.show()

## 3. ORA vs GSEA — a caution (Chapter 4)
ORA depends on your DEG **cutoff** and your **background** gene set; GSEA does not. Different tools/ontologies give different terms. Always report your gene-set version, background, and FDR — reproducibility matters.

### ✏️ Your turn
Change the Jaccard threshold from `0.25` to `0.1` and redraw. What happens to the number of edges and clusters?

In [ ]:
# your code here


<details><summary>▶ Solution</summary>

Lowering the threshold to `0.1` adds many more edges — pathways that share even a few genes get connected, so the map becomes denser and clusters merge. A higher threshold gives sparser, cleaner themes. There is no single 'correct' value; it is a visualization choice.

</details>

## 🎓 You finished the workshop!
You went from a raw count matrix → differential expression → pathways → an enrichment map, all on real data.

Back to **[Omics Atlas](https://omics.iotok.org)**.